# 01 — Data & Preprocessing

**Part of the 5-notebook split** of `code-mixed-hate-speech-detection-pipeline-v2.ipynb`.
The complete original notebook is kept unchanged for upload.

**Scope:** Setup (0.1–0.3), data acquisition & de-identification (1.1–1.3), EDA and the word-level code-mixing feature definition (2.1–2.5). No models.

**Prerequisites:** Internet ON (pinned HF revision); Kaggle /kaggle/working writable.

**Writes:** prism_{train,val,test}.csv, data_card.json, EDA figures, split_state/run_config.json

> Run 01 → 05 in order in the same environment: Kaggle with **Internet ON** and
> **GPU T4 x2**. Every assigned code cell is copied verbatim from the original;
> only the small handoff cells are new. Outputs are intentionally cleared — the
> executed results live in the unchanged original notebook, the PDF, and the
> `results/` artifacts.

### 0.1 Dependencies

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# transformers>=4.46 required: eval_strategy + processing_class were introduced in 4.46.
pip_install([
    "transformers>=4.46,<5",
    "datasets>=3.0",
    "accelerate>=0.34",
    "scipy>=1.11",
])

### 0.2 Imports & Global Configuration (single source of truth for the environment)

In [ ]:
import os, re, json, math, time, random, hashlib, shutil, warnings, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset as TorchDataset

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed as hf_set_seed,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, classification_report,
)
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 110

# ---- Environment contract: the ONLY place run mode is decided ---------------
SMOKE_TEST = False   # True = tiny data + tiny model, CPU fast pass (~2 min)

PRISM_ID  = "pankajbiswas6/prism-hinglish-hate-speech"
MEHATE_REPO = "l3cube-pune/MarathiNLP"          # GitHub repo, CSV fetched at HEAD SHA
MEHATE_CSV  = "MeEval/L3Cube-MeHate/codemix-hate-small-test.csv"

SEEDS = [42, 43, 44]
TINY_MODEL = "sshleifer/tiny-distilroberta-base"
MODELS = {
    "hing-roberta": "l3cube-pune/hing-roberta",
    "muril":        "google/muril-base-cased",
    "xlm-roberta":  "xlm-roberta-base",
}
if SMOKE_TEST:
    MODELS = {k: TINY_MODEL for k in MODELS}

MAX_LEN     = 128 if not SMOKE_TEST else 32
NUM_EPOCHS  = 3   if not SMOKE_TEST else 1
BATCH_SIZE  = 32  if not SMOKE_TEST else 4
BOOTSTRAP_N = 1_000 if not SMOKE_TEST else 100
SMOKE_N     = 200
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.1   # named, not magic

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
RESULTS_DIR, FIGURES_DIR, PREDS_DIR = WORK_DIR/"results", WORK_DIR/"figures", WORK_DIR/"preds"
for d in (RESULTS_DIR, FIGURES_DIR, PREDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fail fast: the full run is impossible on CPU and would silently drop fp16.
if not SMOKE_TEST:
    assert torch.cuda.is_available(), (
        "GPU required: set Accelerator to 'GPU T4 x2' (or P100) in notebook settings. "
        "For a CPU pass, set SMOKE_TEST = True in section 0.2.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"SMOKE_TEST = {SMOKE_TEST} | Device = {DEVICE} | Work dir = {WORK_DIR.resolve()}")

### 0.3 Reproducibility helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (incl. CUDA). Note: with fp16 + cuDNN this is
    reproducible-in-practice, not bit-exact; `full_determinism=True` would cost speed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

ALL_SEED_RESULTS = []   # one row per (model, seed, split)
PER_CLASS_RESULTS = []  # per-class P/R/F1 rows
COSTS = []              # params / train time / ms-per-item per (model, seed)
THRESH_CURVES = {}      # (model_key, seed) -> (thresholds, val macro-F1 curve)
LOG_HISTORY = {}        # (model_key, seed) -> trainer.state.log_history
TRAINED = {}            # (model_key, seed=SEEDS[0]) -> artifacts needed in sections 6-7

print("Reproducibility helpers ready.")

In [ ]:
# ---- Shared handoff directory for the 5-part notebook chain ------------------
# Run 01 -> 02 -> 03 -> 04 -> 05 in the SAME environment (Kaggle, Internet ON, GPU).
STATE_DIR = WORK_DIR / "split_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
print("Split state dir:", STATE_DIR)

## 1. Data Acquisition & De-identification

### 1.1 Load PRISM at a pinned revision, validate against the card

In [ ]:
# Pin the exact upstream revision so `EXPECTED` row counts cannot drift out from under us.
PRISM_SHA = HfApi().dataset_info(PRISM_ID).sha
print(f"PRISM revision (pinned): {PRISM_SHA}")
raw = load_dataset(PRISM_ID, revision=PRISM_SHA)
available = list(raw.keys())
print("Available splits:", available)

def pick_split(names, wanted):
    # Map a canonical name (train/val/test) onto whatever the repo calls it.
    aliases = {"train": ["train"], "val": ["val", "validation", "dev", "valid"], "test": ["test"]}[wanted]
    for a in aliases:
        if a in names:
            return a
    raise ValueError(f"No split matching {wanted!r} in {names}")

SPLIT_MAP = {k: pick_split(available, k) for k in ("train", "val", "test")}
print("Split map:", SPLIT_MAP)

def to_frame(split_key):
    df = raw[SPLIT_MAP[split_key]].to_pandas()
    df = df.rename(columns={"language": "lang"})   # defensive rename
    need = {"text", "label", "lang"}
    missing = need - set(df.columns)
    assert not missing, f"PRISM {split_key}: missing columns {missing}"
    df = df[["text", "label", "lang"]].copy()
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df

if SMOKE_TEST:
    prism = {k: to_frame(k).head(SMOKE_N).reset_index(drop=True) for k in ("train", "val", "test")}
else:
    prism = {k: to_frame(k).reset_index(drop=True) for k in ("train", "val", "test")}

EXPECTED = {"train": 17_704, "val": 2_950, "test": 8_852}   # verified against revision above
if not SMOKE_TEST:
    for k, n in EXPECTED.items():
        assert len(prism[k]) == n, f"PRISM {k}: {len(prism[k])} rows, expected {n}"
print("PRISM split sizes:", {k: len(v) for k, v in prism.items()})
for k, df in prism.items():
    print(f"  {k:5s} label balance: {df['label'].value_counts(normalize=True).round(3).to_dict()}")

### 1.2 De-identification — handles, URLs, emails, phones, PII-safe elongation collapse

In [ ]:
URL_RE    = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE  = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE  = re.compile(r"(?:\+?\d[\d\s\-()]{7,}\d)")          # >= 8 digits total
USER_RE   = re.compile(r"@\w+")
LONG_RUN  = re.compile(r"([A-Za-z\u0900-\u097F])\1{3,}")       # letters only: 'soooo'->'soo'

def deidentify(text: str) -> str:
    t = URL_RE.sub("<URL>", text)
    t = EMAIL_RE.sub("<EMAIL>", t)
    t = PHONE_RE.sub("<PHONE>", t)
    t = USER_RE.sub("<USER>", t)
    t = LONG_RUN.sub(r"\1\1", t)
    return t.strip()

for name, df in prism.items():
    prism[name]["text"] = df["text"].map(deidentify)

sample = prism["train"].loc[prism["train"]["text"].str.contains("<USER>|<URL>|<EMAIL>|<PHONE>"), "text"].head(3)
print("De-identified samples:")
for s in sample:
    print("  •", s[:110])

### 1.3 Persist the processed splits (checksummed, reproducible)

In [ ]:
def sha256_of_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

for k, df in prism.items():
    out = RESULTS_DIR / f"prism_{k}.csv"
    df.to_csv(out, index=False)
    print(f"{out.name:20s} rows={len(df):6d}  sha256={sha256_of_file(out)[:16]}…")

DATA_CARD = {
    "dataset": PRISM_ID, "revision_sha": PRISM_SHA, "licence": "MIT",
    "splits": {k: len(v) for k, v in prism.items()},
    "smoke_test": SMOKE_TEST,
    "deidentification": "@handle-><USER>, URL-><URL>, EMAIL-><EMAIL>, PHONE-><PHONE>, "
                        "letter elongation normalised, emojis kept",
    "pii_note": "Regex-based; names in plain text are NOT removed. Stated as such in the report.",
}
(RESULTS_DIR / "data_card.json").write_text(json.dumps(DATA_CARD, indent=2))
print("data_card.json written (revision pinned).")

## 2. Exploratory Data Analysis

### 2.1 Class balance per language

In [ ]:
train_df = prism["train"].copy()   # copy: EDA must not mutate the shared splits

fig, ax = plt.subplots(figsize=(7, 4))
ct = pd.crosstab(train_df["lang"], train_df["label"], normalize="index")
ct.plot(kind="bar", ax=ax, rot=0, color=["#4c8bf5", "#e5484d"])
ax.set_title("PRISM train — class share by language")
ax.set_xlabel("language"); ax.set_ylabel("share")
ax.legend(["non-hate (0)", "hate (1)"])
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_class_balance.png"); plt.show()
print(ct.round(3))

### 2.2 Text-length distributions per class

In [ ]:
train_df["n_words"] = train_df["text"].str.split().str.len()

fig, ax = plt.subplots(figsize=(7, 4))
for lab in (0, 1):
    subset = train_df.loc[train_df["label"] == lab, "n_words"]
    ax.hist(subset.clip(upper=80), bins=40, alpha=0.55,
            label=f"{'hate' if lab else 'non-hate'} (n={len(subset)})")
ax.set_title("Word-count distribution by class (clipped at 80)")
ax.set_xlabel("words"); ax.set_ylabel("posts"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_length_dist.png"); plt.show()
print(train_df.groupby("label")["n_words"].describe().round(1))

### 2.3 Most distinctive character n-grams per class

In [ ]:
from collections import Counter

def top_char_ngrams(texts, n=3, k=12):
    counts = Counter()
    for t in texts:
        norm = re.sub(r"\s+", " ", t.lower())
        counts.update(norm[i:i+n] for i in range(len(norm) - n + 1))
    return counts.most_common(k)

for lab, name in [(1, "hate"), (0, "non-hate")]:
    print(f"\nTop char trigrams — {name}:")
    print("  ", top_char_ngrams(train_df.loc[train_df["label"] == lab, "text"]))

### 2.4 Code-mixing statistics — word-level language mixing (v2 metric)

v1 measured **script** (Devanagari ratio), which cannot see Romanised Hinglish at all.
v2 measures **language** per word: Devanagari tokens and tokens in a romanised-Hindi
lexicon count as Hindi; remaining Latin tokens count as English. Reported:

- `hindi_token_ratio` — share of Hindi-language tokens,
- `mixing_index` — minority-language token share `min(hi, en) / n` (0 = monolingual, 0.5 = balanced mix),
- `word_switch_points` — adjacent token pairs changing language (placeholders/punctuation/emoji excluded, so no phantom switches),
- `devanagari_ratio` — kept as a *script* covariate, not as the mixing measure.

*Caveat (stated in the report):* lexicon-based language ID underestimates Hindi for
content words absent from the lexicon; the index is a lower bound on mixing.

In [ ]:
DEVANAGARI = re.compile(r"[\u0900-\u097F]")
TOKEN_RE   = re.compile(r"[A-Za-z\u0900-\u097F]+")
PLACEHOLDER_RE = re.compile(r"<URL>|<USER>|<EMAIL>|<PHONE>")

# Compact romanised-Hindi lexicon: function words, common adverbs and frequent
# (incl. profane) content words, with common transliteration variants.
ROMAN_HINDI = set("""
hai hain he hn nahi nahin nai nhi kya kyu kyun ku aue aur ya yaani mein me mai maiya ka ki ke ko se
par per bhi bi hi to tho toh tu tum tumhari tumhara tumhare hum humara humari apna apni apne aap
yeh ye yehe woh vo wo voh kaha kahan kab kaisa kaise kaisi itna itni jaisa jaise abhi ab kal aaj
log logon logo logo sabh sab sabhi kuch koi bahut bohut bohot bhut thoda thodi zyada jyada kam
acha achha acha achhi accha theek thik karo karna karte kar kiya kiye kardia diya diye liya liye
lena dena jata jaate jana gaya gaye gayi raha rahe rahi hogaya hogae hojayega mila mile milega
wala wali waale vale vala walla banda bande log insan insaan baat baatein baat bat kaam kaam ka
desh deshdrohi watan hindi hindustan urdu muslim hindu sikh christian pakistani bangladeshi
bharat bharatiya mullah mulla maulvi katuwa katua katwe jihadi jihad maza hab IB dat
paisa paise rupaye free bijli sadak paani school hospital naukri berojgari mehangai
gali galia gaali gaaliya galiyan maader mader behen behanchod bhosdi bhosdike bhosda chutiya
chutiye chutiye madarchod behenchod randi harami haramkhor kamina kamine kutta kutte kamina
pagal paagal bewakoof bewakoofon bakwaas bakwas bakwaas faltu ghatiya nich neech jhoota jhooth
chor chchor gaddar dhokha betray saala saale sala saley hijre hijra chakka anpadh gawar
bhookha mara maar marey marenge pitai peet pit jhagda ladai fight phir fir jab tab kabhi
kyunki kyunke liye waha yahan idhar udhar aisa waisa kaisa hoga hogi honge thi the thin
na haan han hn ji nahiin chal chalo aaja aao dekh dekho sun suno ruk jaldi abhi turant
""".split())

def _lang_of(tok: str) -> str:
    if DEVANAGARI.search(tok):
        return "hi"
    return "hi" if tok.lower() in ROMAN_HINDI else "en"

def code_mixing_stats(texts):
    rows = []
    for t in texts:
        cleaned = PLACEHOLDER_RE.sub(" ", t)
        toks = TOKEN_RE.findall(cleaned)
        n = len(toks)
        chars = re.sub(r"\s+", "", t)
        devi = len(DEVANAGARI.findall(chars)) / len(chars) if chars else 0.0
        if n == 0:
            rows.append((0.0, 0.0, 0, devi))
            continue
        langs = [_lang_of(x) for x in toks]
        hi = sum(1 for l in langs if l == "hi")
        hindi_ratio = hi / n
        minority = min(hi, n - hi) / n
        switches = sum(1 for a, b in zip(langs, langs[1:]) if a != b)
        rows.append((hindi_ratio, minority, switches, devi))
    return pd.DataFrame(rows, columns=["hindi_token_ratio", "mixing_index",
                                       "word_switch_points", "devanagari_ratio"])

mix = code_mixing_stats(train_df["text"])
train_df = pd.concat([train_df.reset_index(drop=True), mix], axis=1)
print(train_df[["hindi_token_ratio", "mixing_index", "word_switch_points", "devanagari_ratio"]]
      .describe().round(3))
print("\nMean by language:")
print(train_df.groupby("lang")[["hindi_token_ratio", "mixing_index",
                                "word_switch_points", "devanagari_ratio"]].mean().round(3))

### 2.5 Figure — mixing index and switch points by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lab, col in [(0, "#4c8bf5"), (1, "#e5484d")]:
    axes[0].hist(train_df.loc[train_df["label"] == lab, "mixing_index"],
                 bins=40, alpha=0.55, color=col, label=f"{'hate' if lab else 'non-hate'}")
    axes[1].hist(train_df.loc[train_df["label"] == lab, "word_switch_points"],
                 bins=30, alpha=0.55, color=col, label=f"{'hate' if lab else 'non-hate'}")
axes[0].set_title("Language mixing index by class"); axes[0].set_xlabel("mixing index (minority token share)")
axes[1].set_title("Word-level code-switch points by class"); axes[1].set_xlabel("switches per post")
for ax in axes:
    ax.set_ylabel("posts"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_code_mixing.png"); plt.show()

print("Mean mixing index by (lang, label):")
print(train_df.groupby(["lang", "label"])["mixing_index"].mean().round(3).unstack())

In [ ]:
# ---- Split handoff (01 -> 02..05) --------------------------------------------
_run_cfg = {
    "prism_id": PRISM_ID, "prism_sha": PRISM_SHA, "seeds": SEEDS,
    "smoke_test": SMOKE_TEST, "max_len": MAX_LEN, "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE, "bootstrap_n": BOOTSTRAP_N,
    "models": MODELS, "mehate_repo": MEHATE_REPO, "mehate_csv": MEHATE_CSV,
}
(STATE_DIR / "run_config.json").write_text(json.dumps(_run_cfg, indent=2))
for _k in ("train", "val", "test"):
    assert (RESULTS_DIR / f"prism_{_k}.csv").exists()
print("Handoff written:", STATE_DIR / "run_config.json")
print("Processed splits:", {k: len(prism[k]) for k in ("train", "val", "test")})